In [5]:
!pip install ultralytics


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [6]:
import csv
import cv2
from datetime import datetime
from ultralytics import YOLO

# ==========================================
# 1. Configurare Model
# ==========================================
model = YOLO('yolov8n.pt')

# ==========================================
# 2. Configurare Fişiere Video
# ==========================================
input_video_path = 'street.mp4'
output_video_path = 'rezultat_detectie.mp4'

cap = cv2.VideoCapture(input_video_path)

if not cap.isOpened():
    print(f"Eroare: Nu s-a putut deschide fișierul {input_video_path}.")
else:
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    # ==========================================
    # CSV Logging Setup
    # ==========================================
    CSV_FILE = "raport_trafic.csv"
    with open(CSV_FILE, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Timp", "Masini", "Pietoni"])

    LOG_INTERVAL = fps if fps > 0 else 30  # o dată pe secundă
    frames_processed = 0

    print(f"Procesare începută! Total cadre: {total_frames}")

    # ==========================================
    # 3. Bucla de Procesare
    # ==========================================
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frames_processed += 1

        # Detectăm Persoane(0), Mașini(2) și Motociclete(3)
        results = model(frame, classes=[0, 2, 3], conf=0.4, verbose=False)

        # ==========================================
        # 4. Numărăm obiectele pe categorii
        # ==========================================
        numar_masini = 0
        numar_pietoni = 0
        if results[0].boxes is not None:
            for box in results[0].boxes:
                cls = int(box.cls[0])
                if cls == 0:
                    numar_pietoni += 1
                elif cls in [2, 3]:
                    numar_masini += 1

        # Desenăm box-urile YOLO
        annotated_frame = results[0].plot()

        # ==========================================
        # 5. HUD — afișăm numărătoarea pe ecran
        # ==========================================
        hud_x, hud_y = 20, 120   # poziție: mai jos față de margine

        overlay = annotated_frame.copy()
        cv2.rectangle(overlay, (hud_x - 10, hud_y - 10), (hud_x + 330, hud_y + 100), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.55, annotated_frame, 0.45, 0, annotated_frame)

        cv2.putText(annotated_frame, f"Masini:  {numar_masini}", (hud_x, hud_y + 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.3, (0, 200, 255), 3)
        cv2.putText(annotated_frame, f"Pietoni: {numar_pietoni}", (hud_x, hud_y + 85),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.3, (0, 255, 100), 3)

        # ==========================================
        # 6. Salvare CSV o dată pe secundă
        # ==========================================
        if frames_processed % LOG_INTERVAL == 0:
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            with open(CSV_FILE, "a", newline="") as f:
                writer = csv.writer(f)
                writer.writerow([timestamp, numar_masini, numar_pietoni])

        out.write(annotated_frame)

        if frames_processed % 10 == 0:
            print(f"Procesat: {frames_processed}/{total_frames} cadre", end="\r")

    # ==========================================
    # 7. Eliberare Resurse
    # ==========================================
    cap.release()
    out.release()
    print(f"\n[SUCCES] Video salvat: {output_video_path}")
    print(f"[SUCCES] Date salvate în: {CSV_FILE}")
    print("Descarcă ambele fișiere din meniul din stânga.")

Procesare începută! Total cadre: 1706
Procesat: 1700/1706 cadre
[SUCCES] Video salvat: rezultat_detectie.mp4
[SUCCES] Date salvate în: raport_trafic.csv
Descarcă ambele fișiere din meniul din stânga.
